In [ ]:
# Puxa (ou atualiza) o repositório com data/clean/dataset.jsonl, configs/ e src/, e autentica
# no Hugging Face. GH_TOKEN e HF_TOKEN vêm do Colab Secrets (ícone de chave na sidebar).
import os
from google.colab import userdata
from huggingface_hub import login

GH_TOKEN = userdata.get("GH_TOKEN")
GH_USER = "devlucascfarias"
REPO_NAME = "Conatus-Phronesis"
BRANCH = "phronesis-thinking"
REPO = f"/content/{REPO_NAME}"
REPO_URL = f"https://{GH_TOKEN}@github.com/{GH_USER}/{REPO_NAME}.git"

if os.path.isdir(f"{REPO}/.git"):
    !cd {REPO} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO}

!cd {REPO} && git branch --show-current

del GH_TOKEN, REPO_URL

login(token=userdata.get("HF_TOKEN"))


In [ ]:
%pip install -q unsloth
%pip install -q --no-deps trl peft accelerate bitsandbytes
import torch
print(torch.cuda.get_device_name(0))


In [ ]:
# Espelha configs/train_config.yaml. Base: IBM Granite 4.1 (migrado do Qwen3-14B em 2026-07-28)
# — troca de família: chat template, tokenizer e canal <think> mudam todos.
# Rodar PRIMEIRO com USE_GUINEA_PIG=True (3B, barato) para validar o pipeline.
USE_GUINEA_PIG = True
MODEL_SIZE = "8b"

_MODEL_BY_SIZE = {
    "3b":  "ibm-granite/granite-4.1-3b",
    "8b":  "ibm-granite/granite-4.1-8b",   # alvo atual
    "30b": "ibm-granite/granite-4.1-30b",
}

MODEL = "ibm-granite/granite-4.1-3b" if USE_GUINEA_PIG else _MODEL_BY_SIZE[MODEL_SIZE]
TAG = "3b_pig" if USE_GUINEA_PIG else f"granite{MODEL_SIZE}"

# Medido com o tokenizer do Granite sobre os 1206 exemplos: máximo 1456 tokens, p99 1308,
# mediana 610 — zero descartes em 2048.
MAX_SEQ_LEN = 2048
TRAIN_FILE = f"{REPO}/data/clean/train.jsonl"

# Um 8B dense em QLoRA cabe com micro-batch 2 no L4 e no T4; efetivo ~16 pelo acúmulo.
BATCH_SIZE = 2
GRAD_ACCUM = 16 // BATCH_SIZE

import torch
print(f"GPU: {torch.cuda.get_device_name(0)} "
      f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
print(f"MODEL={MODEL}  MAX_SEQ_LEN={MAX_SEQ_LEN}  batch={BATCH_SIZE} x accum={GRAD_ACCUM}")


In [ ]:
# Renderiza dataset.jsonl no chat template do Granite + máscara de loss (seção 4.3).
# CONFIRA a saída: '(N descartados por exceder ... tokens)' tem que vir com N=0.
!cd {REPO} && python src/build_dataset.py data/clean/dataset.jsonl --out data/clean/train.jsonl --model {MODEL} --max-len {MAX_SEQ_LEN}


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
# O tokenizer do Granite vem com padding_side='left' — correto para inferência, errado para
# treino (desloca os labels).
tokenizer.padding_side = "right"

# Se o Unsloth não tiver caminho para GraniteForCausalLM, plano B: AutoModelForCausalLM +
# BitsAndBytesConfig(load_in_4bit=True) + peft.get_peft_model com os mesmos target_modules e
# gradient_checkpointing=True no SFTConfig. É o que a rodada com a cobaia 3B decide.
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


In [ ]:
from datasets import load_dataset

# train.jsonl já vem do build_dataset.py com o campo 'text' renderizado via
# tokenizer.apply_chat_template (nunca concatenar strings na mão).
full = load_dataset("json", data_files=TRAIN_FILE, split="train")
dataset = full.remove_columns([c for c in full.column_names if c != "text"])
print(f"treino: {len(dataset)}")
print(dataset[0]["text"][:600])


In [ ]:
import inspect

from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

# neftune_noise_alpha só existe a partir da transformers 4.35 — detecta em runtime em vez de
# assumir a versão do Colab.
extra = {}
if "neftune_noise_alpha" in set(inspect.signature(SFTConfig.__init__).parameters):
    extra["neftune_noise_alpha"] = 5

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        logging_steps=5,
        seed=42,
        output_dir=f"outputs_{TAG}",
        report_to="none",
        **extra,
    ),
)

# Loss só nos turnos do assistant. Marcadores do Granite 4.1 — os mesmos de
# src/build_dataset.py. Se estiverem errados a máscara zera ou engole o turno seguinte SEM
# erro: por isso a célula seguinte é obrigatória.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_of_role|>user<|end_of_role|>",
    response_part="<|start_of_role|>assistant<|end_of_role|>",
)

print(f"~{(len(dataset) // 16) * 2} passos de otimização")


In [ ]:
# A saída tem que: começar no conteúdo do assistant (tipicamente '<think>'), terminar em
# '<|end_of_text|>', e não conter texto do usuário nem de <tool_response>.
sample = trainer.train_dataset[0]
trainable = [t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100]
print("TREINÁVEL:", tokenizer.decode(trainable)[:500])
assert trainable, "máscara zerou tudo — conferir marcadores do template"


In [ ]:
stats = trainer.train()
print(stats)


In [ ]:
model.save_pretrained(f"{REPO}/outputs/adapter_{TAG}")
tokenizer.save_pretrained(f"{REPO}/outputs/adapter_{TAG}")
model.save_pretrained_merged(f"{REPO}/outputs/merged_{TAG}", tokenizer, save_method="merged_16bit")
print("Salvo.")


In [ ]:
# Sanidade: o checkpoint carrega e gera no template correto.
import json

FastLanguageModel.for_inference(model)
tools = json.load(open(f"{REPO}/configs/tools.json", encoding="utf-8"))["tools"]
msgs = [{"role": "user", "content": "Quanto tá o dólar hoje?"}]
# Sem enable_thinking: no Granite o <think> vem do treino, não do template.
prompt = tokenizer.apply_chat_template(msgs, tools=tools, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=2048, do_sample=False,
                     repetition_penalty=1.15, no_repeat_ngram_size=8,
                     pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
